In [156]:
import pandas as pd
import numpy as np

# 1. Load the Core Files
results = pd.read_csv('data/MRegularSeasonDetailedResults.csv')
teams = pd.read_csv('data/MTeams.csv')
seeds = pd.read_csv('data/MNCAATourneySeeds.csv')

# 2. Clean Seeds: Convert 'W01' -> 1 (Integer)
# This allows the model to calculate "Seed Difference" mathematically.
seeds['Seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))

# 3. Create Season-Long Averages (Including Efficiency Ingredients)
# We need FGA and FGM3 to calculate Effective Field Goal Percentage (eFG%)
cols = ['Score', 'FGM', 'FGA', 'FGM3', 'Ast']
w_rename = {f'W{c}': c for c in cols}
l_rename = {f'L{c}': c for c in cols}

winning_stats = results.groupby(['Season', 'WTeamID'])[list(w_rename.keys())].mean().rename(columns=w_rename)
losing_stats = results.groupby(['Season', 'LTeamID'])[list(l_rename.keys())].mean().rename(columns=l_rename)

# Combine and group to get one row per team per year
team_stats = pd.concat([winning_stats, losing_stats]).groupby(level=[0, 1]).mean()

# 4. Calculate v5 Efficiency Metrics
# eFG% = (FGM + 0.5 * FGM3) / FGA (This rewards 3-point shooting efficiency)
team_stats['eFG'] = (team_stats['FGM'] + 0.5 * team_stats['FGM3']) / team_stats['FGA']
# PPS = Points Per Shot (Measures overall scoring lethality)
team_stats['PPS'] = team_stats['Score'] / team_stats['FGA']
# DefEff Proxy (Points allowed per shot attempt - lower is better)
team_stats['DefEff'] = team_stats['Score'] / (team_stats['FGM'] + 1)

# 5. Extract the Most Recent Season for the 2026 Bracket
latest_season = team_stats.index.get_level_values(0).max()
latest_stats = team_stats.xs(latest_season, level=0)

print(f"✅ Step 1 Complete: Data & Efficiency Metrics prepared for Season {latest_season}.")

✅ Step 1 Complete: Data & Efficiency Metrics prepared for Season 2026.


In [157]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Prepare Training Data
train_results = results.merge(seeds, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'WSeed'}).drop('TeamID', axis=1)
train_results = train_results.merge(seeds, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'LSeed'}).drop('TeamID', axis=1)

# 2. Build the Features
side_a = pd.DataFrame()
side_a['ScoreDiff'] = train_results['WScore'] - train_results['LScore']
side_a['FGM_Diff'] = train_results['WFGM'] - train_results['LFGM']
side_a['Ast_Diff'] = train_results['WAst'] - train_results['LAst']
side_a['SeedDiff'] = train_results['WSeed'] - train_results['LSeed']
side_a['Result'] = 1

side_b = pd.DataFrame()
side_b['ScoreDiff'] = train_results['LScore'] - train_results['WScore']
side_b['FGM_Diff'] = train_results['LFGM'] - train_results['WFGM']
side_b['Ast_Diff'] = train_results['LAst'] - train_results['WAst']
side_b['SeedDiff'] = train_results['LSeed'] - train_results['WSeed']
side_b['Result'] = 0

train_df = pd.concat([side_a, side_b]).dropna()

# 3. Final Training (Using .values to strip names and prevent warnings)
scaler = StandardScaler()
features = ['ScoreDiff', 'FGM_Diff', 'Ast_Diff', 'SeedDiff']

# THE KEY FIX: We use .values here so the model doesn't expect names later
X = scaler.fit_transform(train_df[features].values)
y = train_df['Result'].values

model = LogisticRegression(C=0.001)
model.fit(X, y)

print("✅ Cell 2 Complete: Model trained on raw arrays (No more name warnings).")

✅ Cell 2 Complete: Model trained on raw arrays (No more name warnings).


In [158]:
def predict_game_v5(team1_name, team2_name, seed1, seed2):
    try:
        t1_id = teams[teams['TeamName'] == team1_name]['TeamID'].values[0]
        t2_id = teams[teams['TeamName'] == team2_name]['TeamID'].values[0]

        stats1 = latest_stats.loc[t1_id]
        stats2 = latest_stats.loc[t2_id]

        input_data = [[
            stats1['Score'] - stats2['Score'],
            stats1['FGM'] - stats2['FGM'],
            stats1['Ast'] - stats2['Ast'],
            seed1 - seed2
        ]]

        X_scaled = scaler.transform(input_data)
        stat_prob = model.predict_proba(X_scaled)[0][1]

        # --- THE POWER BLEND (50% Stats / 50% Historical Seed) ---
        # This prevents Mid-Major stat inflation from over-influencing the result
        seed_diff = seed2 - seed1
        seed_only_prob = 0.5 + (seed_diff * 0.05)

        final_prob = (stat_prob * 0.50) + (seed_only_prob * 0.50)

        # --- STRENGTH OF SCHEDULE (SOS) ADJUSTMENT ---
        # We provide a 7.5% adjustment factor for elite seeds (1-6)
        # when facing high-stat Mid-Majors (11-15) to account for SOS variance.
        sos_adjustment = 0.075

        if seed1 <= 6 and seed2 >= 11:
            final_prob += sos_adjustment
        elif seed2 <= 6 and seed1 >= 11:
            final_prob -= sos_adjustment

        return max(0.05, min(0.95, final_prob))

    except Exception as e:
        return 0.5001

In [159]:
# 1. THE OFFICIAL 2026 WINNERS
actual_winners = [
    "Duke", "TCU", "St John's", "Kansas", "Louisville", "Michigan St", "UCLA", "Connecticut",
    "Arizona", "Utah St", "High Point", "Arkansas", "Texas", "Gonzaga", "Miami FL", "Purdue",
    "Michigan", "St Louis", "Texas Tech", "Alabama", "Tennessee", "Virginia", "Kentucky", "Iowa St",
    "Florida", "Iowa", "Vanderbilt", "Nebraska", "VCU", "Illinois", "Texas A&M", "Houston"
]

# 2. MATCHUP DATA (Names matched exactly to your Translator/MTeams keys)
bracket_matchups = [
    ("Duke", "Siena", 1, 16), ("Ohio St", "TCU", 8, 9),
    ("St John's", "Northern Iowa", 5, 12), ("Kansas", "Cal Baptist", 4, 13),
    ("Louisville", "South Florida", 6, 11), ("Michigan St", "N Dakota St", 3, 14),
    ("UCLA", "UCF", 7, 10), ("Connecticut", "Furman", 2, 15),
    ("Arizona", "LIU Brooklyn", 1, 16), ("Villanova", "Utah St", 8, 9),
    ("Wisconsin", "High Point", 5, 12), ("Arkansas", "Hawaii", 4, 13),
    ("BYU", "Texas", 6, 11), ("Gonzaga", "Kennesaw", 3, 14),
    ("Miami FL", "Missouri", 7, 10), ("Purdue", "Queens NC", 2, 15),
    ("Michigan", "Howard", 1, 16), ("Georgia", "St Louis", 8, 9),
    ("Texas Tech", "Akron", 5, 12), ("Alabama", "Hofstra", 4, 13),
    ("Tennessee", "Miami OH", 6, 11), ("Virginia", "Wright St", 3, 14),
    ("Kentucky", "Santa Clara", 7, 10), ("Iowa St", "Tennessee St", 2, 15),
    ("Florida", "Prairie View", 1, 16), ("Clemson", "Iowa", 8, 9),
    ("Vanderbilt", "McNeese St", 5, 12), ("Nebraska", "Troy", 4, 13),
    ("North Carolina", "VCU", 6, 11), ("Illinois", "Penn", 3, 14),
    ("St Mary's CA", "Texas A&M", 7, 10), ("Houston", "Idaho", 2, 15)
]

# 3. FORMATTED OUTPUT ENGINE
correct_predictions = 0
total_games = 0

print(f"{'GAME MATCHUP':<40} | {'CONF.':<7} | {'PREDICTED':<15} | {'ACTUAL':<15} | {'STATUS'}")
print("-" * 105)

for i, (t1, t2, s1, s2) in enumerate(bracket_matchups):
    prob = predict_game_v5(t1, t2, s1, s2)

    # We now catch the errors properly
    total_games += 1
    predicted_winner = t1 if prob > 0.5 else t2
    conf = prob if prob > 0.5 else (1 - prob)
    actual_winner = actual_winners[i]

    is_correct = (predicted_winner == actual_winner)
    if is_correct:
        correct_predictions += 1
        status = "✅"
    else:
        status = "❌"

    matchup_text = f"{t1} vs {t2}"
    print(f"{matchup_text:<40} | {conf:>6.1%} | {predicted_winner:<15} | {actual_winner:<15} | {status}")

if total_games > 0:
    accuracy = (correct_predictions / total_games) * 100
    print("-" * 105)
    print(f"🏆 FINAL MODEL ACCURACY (v5.1): {accuracy:.2f}% ({correct_predictions}/{total_games} games)")

GAME MATCHUP                             | CONF.   | PREDICTED       | ACTUAL          | STATUS
---------------------------------------------------------------------------------------------------------
Duke vs Siena                            |  95.0% | Duke            | Duke            | ✅
Ohio St vs TCU                           |  52.1% | Ohio St         | TCU             | ❌
St John's vs Northern Iowa               |  92.4% | St John's       | St John's       | ✅
Kansas vs Cal Baptist                    |  93.1% | Kansas          | Kansas          | ✅
Louisville vs South Florida              |  66.7% | Louisville      | Louisville      | ✅
Michigan St vs N Dakota St               |  90.2% | Michigan St     | Michigan St     | ✅
UCLA vs UCF                              |  54.0% | UCLA            | UCLA            | ✅
Connecticut vs Furman                    |  95.0% | Connecticut     | Connecticut     | ✅
Arizona vs LIU Brooklyn                  |  95.0% | Arizona         | Arizona 

In [161]:
# ROUND OF 32 PREDICTIONS ---
# Names updated to match MTeams.csv exactly for the Predictor to work

round_32_matchups = [
    # East Region
    ("Duke", "TCU", 1, 9),
    ("St John's", "Kansas", 5, 4),
    ("Louisville", "Michigan St", 6, 3),
    ("UCLA", "Connecticut", 7, 2),

    # West Region
    ("Arizona", "Utah St", 1, 9),
    ("Vanderbilt", "Nebraska", 5, 4),
    ("Miami FL", "Purdue", 7, 2),
    ("Texas", "Gonzaga", 11, 3),

    # Midwest Region
    ("Michigan", "St Louis", 1, 9),
    ("Texas Tech", "Alabama", 5, 4),
    ("Tennessee", "Virginia", 6, 3),
    ("Kentucky", "Iowa St", 7, 2),

    # South Region
    ("Florida", "Iowa", 1, 9),
    ("Arkansas", "High Point", 4, 12),
    ("VCU", "Illinois", 11, 3),
    ("Texas A&M", "Houston", 10, 2)
]

print(f"{'2026 ROUND OF 32 MATCHUP':<45} | PREDICTED WINNER (CONF.)")
print("-" * 80)

for t1, t2, s1, s2 in round_32_matchups:
    # Using your optimized v5 predictor
    prob = predict_game_v5(t1, t2, s1, s2)

    if prob is not None:
        winner = t1 if prob > 0.5 else t2
        # Calculate confidence based on which side of 50% we landed
        conf = prob if prob > 0.5 else (1 - prob)

        matchup_label = f"{t1} ({s1}) vs {t2} ({s2})"
        print(f"{matchup_label:<45} | {winner:<15} ({conf:.1%})")

2026 ROUND OF 32 MATCHUP                      | PREDICTED WINNER (CONF.)
--------------------------------------------------------------------------------
Duke (1) vs TCU (9)                           | Duke            (79.3%)
St John's (5) vs Kansas (4)                   | St John's       (54.4%)
Louisville (6) vs Michigan St (3)             | Louisville      (52.2%)
UCLA (7) vs Connecticut (2)                   | Connecticut     (69.8%)
Arizona (1) vs Utah St (9)                    | Arizona         (82.0%)
Vanderbilt (5) vs Nebraska (4)                | Vanderbilt      (58.9%)
Miami FL (7) vs Purdue (2)                    | Purdue          (69.9%)
Texas (11) vs Gonzaga (3)                     | Gonzaga         (83.2%)
Michigan (1) vs St Louis (9)                  | Michigan        (79.4%)
Texas Tech (5) vs Alabama (4)                 | Alabama         (65.7%)
Tennessee (6) vs Virginia (3)                 | Virginia        (58.9%)
Kentucky (7) vs Iowa St (2)                   | Iowa S